# Database initialization

In [ ]:
from db_management_tools import DataLoader
dataset_dir = r"D:\AIC26"
data_loader = DataLoader(dataset_dir)

# Upload data

In [ ]:
data_loader.create_and_fill_database(
    "aic2026_upload_test",
    upload_batch_size=256,
    upload_num_workers=16
)

# Query initialization

In [1]:
from db_management_tools import QueryManager
from models import SigLIP2Model, BGEM3Model
from peripheral_tools import NotebookDisplay

image_text_feature_extractor = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")
text_text_feature_extractor = BGEM3Model(name="BAAI/bge-m3", device="cuda")

query_manager = QueryManager(
    image_text_feature_extractor=image_text_feature_extractor,
    text_text_feature_extractor=text_text_feature_extractor
)
display = NotebookDisplay(315, 5)

[INFO] Loading quantized google/siglip2-so400m-patch14-384
[INFO] google/siglip2-so400m-patch14-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


In [2]:
query = "Đoạn phim bắt đầu bằng một bản đồ, trên đó một loại công trình thủy lợi lần lượt xuất hiện bốn lần. Sau đó chuyển sang cảnh một con đập được quay từ trên cao, tiếp đến là cảnh cận con đập dưới trời mưa."
limit = 100
points = query_manager.aggregated_search_v2(
    "aic2026",
    query=query,
    search_limit=limit,
    aggregation_neighbor_limit=10,
    time_distance_threshold=30,
    use_time_distance=True,
    entropy_based_bonus=True,
    agg_z_score_sigmoid_norm=True,
    aggregation_weight=0.5,
    num_workers=32,
    logging=True,
)

# print("\nList of candidates:")
# for point in points: print(f"{point.payload["vid_id"]}_{point.payload["frame_id"]}")
# display.display(points)

[np.float64(0.00041079231545559924), np.float64(0.0013115829451159078), np.float64(0.0002847842455653282), np.float64(0.0008088802249256455), np.float64(0.0004877637329444573), np.float64(0.0003211943108877291), np.float64(0.0002168145365817331), np.float64(0.00047157835470668097), np.float64(0.0018695414145784017), np.float64(0.000766776843860506), np.float64(0.0008745201355795373), np.float64(0.0010636235544385642), np.float64(0.0003093865880113844), np.float64(0.002594645511392768), np.float64(0.0016236711691192118), np.float64(0.0016519715332063978), np.float64(0.001045746326993724), np.float64(0.000613515032653722), np.float64(0.0006019400364147831), np.float64(0.002352888608397108), np.float64(0.0016779660885347658), np.float64(0.004974500227017176), np.float64(0.00028572002987270473), np.float64(0.0027208610012539047), np.float64(0.0010611345120165039), np.float64(0.0016777693644970393), np.float64(0.000769012388269763), np.float64(0.004757600076868762), np.float64(0.00358389646

# Benchmark

In [ ]:
from peripheral_tools import Benchmarker, BenchmarkerV2

test_settings = [
    # {
    #     "using_vectors": ["siglip2_dense"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_sparse"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_dense"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_dense", "bgem3_sparse"]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.5,
    #     "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    # },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.5,
    #     "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    # },
]

benchmarker = BenchmarkerV2(query_manager)
res = benchmarker.do_benchmark(
    "aic26_mini",
    test_questions=r"C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\question_bank\questions_with_answers.csv",
    test_settings=test_settings,
    save_record=False
)